# Mean Shift Ablation Study for State Model

**Author:** Sid Potti  


## Objective

This notebook tests whether a simple **mean shift baseline** can match the performance of the full State transformer model. The goal is to understand:

1. **How much predictive power comes from learning the "average shift" per perturbation?**
2. **At what training epoch does the transformer surpass the mean shift baseline?**
3. **Can we initialize training with mean shifts to speed up convergence?**

## Hypothesis

The perturbation effect can be approximated as a **consistent shift in embedding space** for each (cell_type, perturbation) pair:

```
perturbed_embedding ≈ control_embedding + mean_shift
```

If this approximation is good, the mean shift baseline should achieve similar MMD scores to the State transformer.

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
from tqdm import tqdm

# Import our mean shift implementation
from mean_shift_ablation import MeanShiftTable, create_pred_h5ad_for_mmd

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Imports successful!")

## Data Configuration

Specify the paths to your training and test data, and the column names in your AnnData objects.

In [ ]:

# Data paths (relative to mean_shift/ directory)
REAL_H5AD_PATH = "data/real.h5ad"                    # Ground truth perturbed cells
CONTROL_DATA_PATH = "data/control_dataset"           # Directory with 30 partitioned control files
PRED_H5AD_PATH_STATE = "data/pred.h5ad"             # State model predictions (for comparison)

# Column names in adata.obs - PARSE data specific
CELL_TYPE_COL = "donor"            # Biological context for grouping (donor in PARSE data)
PERT_COL = "cytokine"              # Column containing perturbation/treatment names
CONTROL_PERT = "PBS"               # Name of control/untreated perturbation

# Embedding location
EMBED_KEY = "tahoe_x1_3b"          # Key in adata.obsm for embeddings, or None for adata.X

# Output paths
OUTPUT_DIR = Path("mean_shift_results")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Output file for MMD evaluation
PRED_H5AD_PATH = OUTPUT_DIR / "pred_lms.h5ad"  # LMS = Latent Mean Shift

print(f"Configuration:")
print(f"  Real data (perturbed): {REAL_H5AD_PATH}")
print(f"  Control data (partitioned): {CONTROL_DATA_PATH}")
print(f"  State predictions: {PRED_H5AD_PATH_STATE}")
print(f"  Output pred file: {PRED_H5AD_PATH}")
print(f"  Cell type column: {CELL_TYPE_COL}")
print(f"  Perturbation column: {PERT_COL}")
print(f"  Control perturbation: {CONTROL_PERT}")
print(f"  Embedding key: {EMBED_KEY}")


## Step 1: Load and Inspect Test Data (real.h5ad)

First, let's load real.h5ad and verify its structure.

In [ ]:
print("Loading real.h5ad (test data with perturbed cells)...")
adata_real = sc.read_h5ad(REAL_H5AD_PATH)

print(f"\nReal data shape: {adata_real.shape}")
print(f"  - Cells: {adata_real.n_obs:,}")
print(f"  - Features: {adata_real.n_vars:,}")

print(f"\nColumns in adata.obs: {list(adata_real.obs.columns)}")
print(f"Keys in adata.obsm: {list(adata_real.obsm.keys())}")

# Get embeddings
if EMBED_KEY and EMBED_KEY in adata_real.obsm:
    real_embeddings = adata_real.obsm[EMBED_KEY]
    print(f"\nUsing embeddings from adata.obsm['{EMBED_KEY}']")
else:
    real_embeddings = adata_real.X.toarray() if hasattr(adata_real.X, 'toarray') else adata_real.X
    print(f"\nUsing embeddings from adata.X")

print(f"Embedding shape: {real_embeddings.shape}")
embedding_dim = real_embeddings.shape[1]

# Verify required columns exist
assert CELL_TYPE_COL in adata_real.obs.columns, f"Column '{CELL_TYPE_COL}' not found in adata.obs"
assert PERT_COL in adata_real.obs.columns, f"Column '{PERT_COL}' not found in adata.obs"
assert 'ctrl_cell_barcode' in adata_real.obs.columns, "Column 'ctrl_cell_barcode' not found in adata.obs"

# Get unique values
unique_cell_lines = adata_real.obs[CELL_TYPE_COL].unique()
unique_perts = adata_real.obs[PERT_COL].unique()

print(f"\nUnique cell lines: {len(unique_cell_lines)}")
print(f"Cell lines: {list(unique_cell_lines)}")

print(f"\nUnique perturbations: {len(unique_perts)}")
print(f"First 10 perturbations: {list(unique_perts[:10])}")

# Check for control perturbation
assert CONTROL_PERT in unique_perts, f"Control perturbation '{CONTROL_PERT}' not found in data"
n_control = (adata_real.obs[PERT_COL] == CONTROL_PERT).sum()
print(f"\nControl cells ('{CONTROL_PERT}'): {n_control:,} ({n_control/adata_real.n_obs*100:.1f}%)")

## Step 2: Compute Mean Shift Table from real.h5ad

For each (donor, cytokine) pair, we compute:

```python
control_mean = mean(control_embeddings for this donor)
perturbed_mean = mean(perturbed_embeddings for this donor + cytokine)
shift = perturbed_mean - control_mean
```

This shift vector captures the "average effect" of applying that cytokine perturbation to cells from that donor.

In [ ]:
print("Computing mean shift table from real.h5ad...")

shift_table = MeanShiftTable()
shift_table.compute_from_anndata(
    adata_path=REAL_H5AD_PATH,
    control_pert=CONTROL_PERT,
    cell_type_col=CELL_TYPE_COL,
    pert_col=PERT_COL,
    embed_key=EMBED_KEY
)

# Save the shift table
shift_table_path = OUTPUT_DIR / "mean_shift_table.pkl"
shift_table.save(str(shift_table_path))

print(f"\nMean shift table saved to: {shift_table_path}")

## Step 3: Verify Shift Table Structure

Let's verify that the computed shifts have the correct dimensions and structure.

In [ ]:
print("="*60)
print("SHIFT TABLE VERIFICATION")
print("="*60)

# Check number of shifts computed
n_shifts = len(shift_table.shifts)
n_control_means = len(shift_table.control_means)

print(f"\nNumber of shifts computed: {n_shifts}")
print(f"Number of cell types with control means: {n_control_means}")

# Verify all shifts have correct dimensions
print(f"\nVerifying shift dimensions...")
for key, shift in shift_table.shifts.items():
    assert shift.shape == (embedding_dim,), f"Shift {key} has wrong shape: {shift.shape}"
print(f"✓ All {n_shifts} shifts have correct shape: ({embedding_dim},)")

# Verify control means have correct dimensions
print(f"\nVerifying control mean dimensions...")
for cell_type, mean in shift_table.control_means.items():
    assert mean.shape == (embedding_dim,), f"Control mean for {cell_type} has wrong shape: {mean.shape}"
print(f"✓ All {n_control_means} control means have correct shape: ({embedding_dim},)")

# Show example shifts
print(f"\nExample shifts:")
for i, (key, shift) in enumerate(list(shift_table.shifts.items())[:3]):
    cell_type, pert = key
    magnitude = np.linalg.norm(shift)
    n_control, n_pert = shift_table.n_samples[key]
    print(f"\n  {i+1}. Cell type: {cell_type}")
    print(f"     Perturbation: {pert}")
    print(f"     Shift magnitude: {magnitude:.4f}")
    print(f"     Sample sizes: {n_control} control, {n_pert} perturbed")
    print(f"     First 5 shift values: {shift[:5]}")

print("\n✓ All assertions passed!")

## Step 4: Analyze Shift Statistics

Let's examine the distribution of shift magnitudes across different perturbations.

In [ ]:
# Get statistics
stats_df = shift_table.get_statistics()

print("Shift magnitude statistics:")
print(stats_df['shift_magnitude'].describe())

print("\nTop 20 perturbations by shift magnitude:")
print(stats_df.head(20))

# Save statistics
stats_path = OUTPUT_DIR / "shift_statistics.csv"
stats_df.to_csv(stats_path, index=False)
print(f"\nStatistics saved to: {stats_path}")

In [ ]:
# Plot distribution of shift magnitudes
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(stats_df['shift_magnitude'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(stats_df['shift_magnitude'].median(), color='r', linestyle='--', 
                label=f'Median: {stats_df["shift_magnitude"].median():.4f}')
axes[0].set_xlabel('Shift Magnitude (L2 Norm)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Distribution of Shift Magnitudes', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Boxplot by cell type
stats_df.boxplot(column='shift_magnitude', by='cell_type', ax=axes[1])
axes[1].set_xlabel('Cell Type', fontsize=12)
axes[1].set_ylabel('Shift Magnitude', fontsize=12)
axes[1].set_title('Shift Magnitudes by Cell Type', fontsize=14)
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shift_magnitude_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Plot saved to: {OUTPUT_DIR / 'shift_magnitude_distribution.png'}")

## Step 5: Create Prediction H5AD File for MMD Evaluation

Now we apply the mean shift to each test cell using **Approach 1**:

**For each perturbed cell in real.h5ad:**
1. Look up `ctrl_cell_barcode` from `.obs`
2. Find that specific control cell's embedding in the control dataset
3. Apply: `pred = control_embedding + mean_shift[(cell_line, perturbation)]`

The output file will have:
- Same cells and metadata as real.h5ad
- Predictions stored in `.obsm['model_preds']`

In [ ]:
create_pred_h5ad_for_mmd(
    adata_test_path=REAL_H5AD_PATH,
    adata_control_path=CONTROL_DATA_PATH,
    shift_table=shift_table,
    output_path=str(PRED_H5AD_PATH),
    control_pert=CONTROL_PERT,
    cell_type_col=CELL_TYPE_COL,
    pert_col=PERT_COL,
    embed_key=EMBED_KEY,
    pred_embed_key="model_preds",  # This is what mmd_anndata_pair.py expects
    ctrl_barcode_col="ctrl_cell_barcode",
    is_partitioned=True  # Enable partitioned loading for 30 files
)

In [ ]:
import subprocess

# MMD evaluation configuration
MMD_OUTPUT_DIR = OUTPUT_DIR / "mmd_evaluation"
MMD_OUTPUT_DIR.mkdir(exist_ok=True)

# Build the command (scripts directory is one level up from mean_shift/)
mmd_command = [
    "python", "../scripts/mmd_state_pipeline/mmd_anndata_pair.py",
    "--adata-real", REAL_H5AD_PATH,
    "--adata-pred", str(PRED_H5AD_PATH),
    "--control-pert", CONTROL_PERT,
    "--pert-col", PERT_COL,
    "--celltype-col", CELL_TYPE_COL,
    "--embed-key", EMBED_KEY,
    "--embed-key-pred", "model_preds",
    "--outdir", str(MMD_OUTPUT_DIR)
]

print("Running MMD evaluation...")
print(f"Command: {' '.join(mmd_command)}\n")

# Run the command
result = subprocess.run(mmd_command, capture_output=True, text=True)

# Print output
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

if result.returncode == 0:
    print(f"\n✓ MMD evaluation complete!")
    print(f"Results saved to: {MMD_OUTPUT_DIR}")
else:
    print(f"\n✗ MMD evaluation failed with return code {result.returncode}")


### What's Done

1. ✓ Loaded real.h5ad (ground truth perturbed embeddings from State predictions)
2. ✓ Computed mean shift table for all (cell_line, perturbation) pairs from real.h5ad
3. ✓ Analyzed shift statistics and visualized shift magnitude distributions
4. ✓ Created `pred_lms.h5ad` with mean shift predictions using **Approach 1**:
   - Used specific control cells (via `ctrl_cell_barcode`) that State paired with each perturbed cell
   - Applied mean shift to those control embeddings: `pred = control_embedding + shift`
5. ✓ Output file is ready for MMD evaluation

### Next Steps

1. **Run MMD evaluation** using the command above to compare:
   - `pred_lms.h5ad` (mean shift baseline) vs `real.h5ad` (ground truth)
   - Download `pred.h5ad` from S3 (State model predictions) and compare vs `real.h5ad`
2. **Compare MMD scores**:
   - If mean shift MMD ≈ State MMD → Simple mean shift captures most perturbation effects
   - If mean shift MMD >> State MMD → Transformer learns complex, context-dependent effects
3. **Analyze per-perturbation and per-cell-line breakdown** to understand where each approach excels


In [ ]:
import json

# Load both MMD summaries
with open(OUTPUT_DIR / "mmd_evaluation/mmd_summary.json", 'r') as f:
    mean_shift_results = json.load(f)

with open(OUTPUT_DIR / "mmd_evaluation_state/mmd_summary.json", 'r') as f:
    state_results = json.load(f)

# Compare key metrics
print("="*80)
print("MMD COMPARISON: Mean Shift Baseline vs State Transformer Model")
print("="*80)

print("\n📊 BASELINE MMD (Control vs Real Perturbed)")
print(f"  Mean Shift - RBF:    {mean_shift_results['baseline_mmd_rbf']['mean']:.6f}")
print(f"  State Model - RBF:   {state_results['baseline_mmd_rbf']['mean']:.6f}")
print(f"  Mean Shift - Energy: {mean_shift_results['baseline_mmd_energy']['mean']:.6f}")
print(f"  State Model - Energy: {state_results['baseline_mmd_energy']['mean']:.6f}")

print("\n🎯 TRANSPORT MMD (Predictions vs Real Perturbed)")
print(f"  Mean Shift - RBF:    {mean_shift_results['transport_mmd_rbf']['mean']:.6f}")
print(f"  State Model - RBF:   {state_results['transport_mmd_rbf']['mean']:.6f}")
print(f"  Mean Shift - Energy: {mean_shift_results['transport_mmd_energy']['mean']:.6f}")
print(f"  State Model - Energy: {state_results['transport_mmd_energy']['mean']:.6f}")

print("\n🔧 CONTROL MMD (Predicted Controls vs Real Controls)")
print(f"  Mean Shift - RBF:    {mean_shift_results['control_mmd_rbf']['mean']:.6f}")
print(f"  State Model - RBF:   {state_results['control_mmd_rbf']['mean']:.6f}")
print(f"  Mean Shift - Energy: {mean_shift_results['control_mmd_energy']['mean']:.6f}")
print(f"  State Model - Energy: {state_results['control_mmd_energy']['mean']:.6f}")

print("\n📈 IMPROVEMENT RATIO (Baseline MMD / Transport MMD)")
print(f"  Mean Shift - RBF:    {mean_shift_results['improvement_ratio_rbf']['mean']:.3f}x")
print(f"  State Model - RBF:   {state_results['improvement_ratio_rbf']['mean']:.3f}x")
print(f"  Mean Shift - Energy: {mean_shift_results['improvement_ratio_energy']['mean']:.6f}x")
print(f"  State Model - Energy: {state_results['improvement_ratio_energy']['mean']:.6f}x")

print("\n" + "="*80)
print("INTERPRETATION")
print("="*80)
print("- Improvement ratio > 1.0: Model improves over baseline (good)")
print("- Improvement ratio < 1.0: Model is worse than baseline (bad)")
print("- Transport MMD closer to Baseline MMD: Model captures drug effects better")
print("="*80)

## Step 7: Compare Mean Shift vs State Model Performance

Let's load and compare the MMD results from both approaches.

In [ ]:
import subprocess

# MMD evaluation for State model predictions
MMD_STATE_OUTPUT_DIR = OUTPUT_DIR / "mmd_evaluation_state"
MMD_STATE_OUTPUT_DIR.mkdir(exist_ok=True)

# Build the command for State model evaluation
mmd_state_command = [
    "python", "../scripts/mmd_state_pipeline/mmd_anndata_pair.py",
    "--adata-real", REAL_H5AD_PATH,
    "--adata-pred", PRED_H5AD_PATH_STATE,  # State model predictions
    "--control-pert", CONTROL_PERT,
    "--pert-col", PERT_COL,
    "--celltype-col", CELL_TYPE_COL,
    "--embed-key", EMBED_KEY,
    "--embed-key-pred", "model_preds",  # State model stores predictions here
    "--outdir", str(MMD_STATE_OUTPUT_DIR)
]

print("Running MMD evaluation for State model predictions...")
print(f"Command: {' '.join(mmd_state_command)}\n")

# Run the command
result = subprocess.run(mmd_state_command, capture_output=True, text=True)

# Print output
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

if result.returncode == 0:
    print(f"\n✓ State model MMD evaluation complete!")
    print(f"Results saved to: {MMD_STATE_OUTPUT_DIR}")
else:
    print(f"\n✗ State model MMD evaluation failed with return code {result.returncode}")

## Step 6: Evaluate State Model Predictions (Baseline Comparison)

Now let's evaluate the State model's predictions (pred.h5ad) vs ground truth to compare against our mean shift baseline.